In [3]:

!pip install langchain langchain-community langchain-huggingface faiss-cpu chromadb pypdf groq -q

In [6]:
!pip install langchain-groq -q
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS, Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.1 MB/s eta 0:00:00


In [7]:
class DocumentLoader:
    def __init__(self, file_path):
        self.file_path = file_path

    def load(self):
        loader = PyPDFLoader(self.file_path)
        docs = loader.load()
        print(f"Loaded {len(docs)} pages")
        return docs

In [8]:
class DocumentChunker:
    def __init__(self):
        self.splitter_small = RecursiveCharacterTextSplitter(
            chunk_size=500, chunk_overlap=100
        )
        self.splitter_large = RecursiveCharacterTextSplitter(
            chunk_size=1000, chunk_overlap=200
        )

    def split(self, docs):
        chunks_small = self.splitter_small.split_documents(docs)
        chunks_large = self.splitter_large.split_documents(docs)

        print(f"Small chunks: {len(chunks_small)}")
        print(f"Large chunks: {len(chunks_large)}")

        return chunks_small, chunks_large

In [9]:
class EmbeddingManager:
    def __init__(self):
        self.embed_large = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2"
        )

        self.embed_small = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

In [18]:
class VectorStoreManager:
    def __init__(self, embed_large, embed_small):
        self.embed_large = embed_large
        self.embed_small = embed_small

    def create_stores(self, chunks_large, chunks_small):
        # FAISS
        self.faiss_db = FAISS.from_documents(chunks_large, self.embed_large)

        # CHROMA
        self.chroma_db = Chroma.from_documents(
            documents=chunks_small,
            embedding=self.embed_small,
            persist_directory="./chroma_db"
        )

        return self.faiss_db, self.chroma_db

In [11]:
class RetrieverManager:
    def __init__(self, faiss_db, chroma_db):
        self.faiss_retriever = faiss_db.as_retriever(search_kwargs={"k": 3})
        self.chroma_retriever = chroma_db.as_retriever(search_kwargs={"k": 3})

    def retrieve(self, query):
        docs1 = self.faiss_retriever.invoke(query)
        docs2 = self.chroma_retriever.invoke(query)
        return docs1 + docs2

In [12]:
class LLMManager:
    def __init__(self, api_key):
        os.environ["GROQ_API_KEY"] = api_key

        self.llm = ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=0.5
        )

    def generate(self, prompt):
        response = self.llm.invoke(prompt)
        return response.content

In [13]:
class PromptManager:
    def __init__(self):
        self.prompt = ChatPromptTemplate.from_template("""
You are a technical assistant for Visual Studio.

Answer ONLY from the given context.
If answer not found, say "Not available in document".

Context:
{context}

Question:
{question}

Answer:
""")

    def format(self, context, question):
        return self.prompt.invoke({
            "context": context,
            "question": question
        })

In [14]:
class RAGPipeline:
    def __init__(self, file_path, groq_api_key):

        loader = DocumentLoader(file_path)
        docs = loader.load()


        chunker = DocumentChunker()
        chunks_small, chunks_large = chunker.split(docs)


        embeddings = EmbeddingManager()


        vector_store = VectorStoreManager(
            embeddings.embed_large,
            embeddings.embed_small
        )
        faiss_db, chroma_db = vector_store.create_stores(
            chunks_large, chunks_small
        )


        self.retriever = RetrieverManager(faiss_db, chroma_db)


        self.llm = LLMManager(groq_api_key)


        self.prompt_manager = PromptManager()

    def format_docs(self, docs):
        return "\n\n".join(doc.page_content for doc in docs)

    def ask(self, query):
        docs = self.retriever.retrieve(query)
        context = self.format_docs(docs)

        final_prompt = self.prompt_manager.format(context, query)
        return self.llm.generate(final_prompt)

In [ ]:
file_path = "/content/technical_document.pdf"
groq_api_key = ""

rag = RAGPipeline(file_path, groq_api_key)

while True:
    query = input("\nAsk your question (type 'exit' to quit): ")

    if query.lower() == "exit":
        break

    answer = rag.ask(query)
    print("\nAnswer:\n", answer)

Loaded 4 pages
Small chunks: 16
Large chunks: 8


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Ask your question (type 'exit' to quit): What is Visual Studio?

Answer:
 Visual Studio is a powerful integrated development environment (IDE) for Windows where you can develop, build, debug, test, and deploy your apps, all in one place.

Ask your question (type 'exit' to quit): How does debugging work?

Answer:
 You can debug your code in Visual Studio by using integrated debugging tools. This allows you to easily debug, profile, and diagnose code. You can step through your code, look at the values stored in variables, set watches on variables to see when values change, and examine the execution path of your code. Visual Studio also provides other ways to debug your code while it runs. 

To learn more about debugging in Visual Studio, you can start by looking at the debugger and also consider debugging with Copilot.

Ask your question (type 'exit' to quit): What are its features?

Answer:
 Visual Studio includes compilers, code completion tools, source control, extensions, and many m